# Applications of Data Science - Summer Term 2026
- Group: 11 - International Taxation
- Group Members:
    - Simon Andreas **ERTL**
    - Berkay **KAYA**
    - Joel **PUTHENPARAMBIL**
    - Fedor **SAMOROKOV**
- Author of this notebook: Simon Andreas **ERTL**
## Group Project: Building an Austrian Tax Law Assistant
This project is split into four parts:
- Creation of a shared corpus regarding Austrian Tax Law from which each group can use for their LLM models. This corpus consists of questions regarding Austrian Tax Law focusing on different topics (e. g. International Taxation), detailed answers to these questions, and sources referenced.
- Applying models to the created corpus. This task itself is split into three parts:
    - Prompt LLMs (Inference): Out-of-the-box Large Language Models should be used to answer the questions created during Task 1.
    - Fine-tune models (Training): The models should then be fine-tuned using additional materials.
    - RAG: Using RAG, the performnance of the models should again be improved.
- Evalution of model performance
- Presentation

This notebook was created in Google Colab using the available T4 GPU architecture. To replicate the results, upload this notebook to Google Colab, and select the right architecture in the top right corner.

### Task 3
Before the tasks itself can be performed, various steps have to be completed beforehand:
- Importing necessary libaries
- Path handling
- Set up of the environment

In the following cell, the filename of the dataset (input and output) can be specified. Only the file name is needed, as the filepath will be handled seperately.

In [3]:
input_dataset = "dataset_clean.csv" # Defining input dataset file name
output_dataset = "output_model3_ERTL.csv" # Defining output dataset file name

In the following code cells, the necessary libraries will be imported.

In [4]:
# Taken from unsloth documentation
!pip install "unsloth[colab-new] @ git+https://github.com/unslothai/unsloth.git"
!pip install --no-deps xformers "trl<0.9.0" peft accelerate bitsandbytes
!pip install pypdf
!pip install sklearn

  Cloning https://github.com/unslothai/unsloth.git to /tmp/pip-install-cnbf7zi5/unsloth_aae6276e4fb448ae83b8a8faf8d9ee1b
  Running command git clone --filter=blob:none --quiet https://github.com/unslothai/unsloth.git /tmp/pip-install-cnbf7zi5/unsloth_aae6276e4fb448ae83b8a8faf8d9ee1b
  Resolved https://github.com/unslothai/unsloth.git to commit 5aa8c152468cd741949c72187dcb0702a1f11f43
  Installing build dependencies ... done
  Getting requirements to build wheel ... done
  Preparing metadata (pyproject.toml) ... done
  Using cached trl-0.24.0-py3-none-any.whl.metadata (11 kB)
Using cached trl-0.24.0-py3-none-any.whl (423 kB)
  Attempting uninstall: trl
    Found existing installation: trl 0.8.6
    Uninstalling trl-0.8.6:
      Successfully uninstalled trl-0.8.6
  Using cached trl-0.8.6-py3-none-any.whl.metadata (11 kB)
Using cached trl-0.8.6-py3-none-any.whl (245 kB)
  Attempting uninstall: trl
    Found existing installation: trl 0.24.0
    Uninstalling trl-0.24.0:
      Successfully 

In [5]:
# Importing libraries
import pandas as pd # For dataframe handling
from pathlib import Path # For path handling
import torch # For back-end
from unsloth import FastLanguageModel # To interact with the models
from google.colab import files # For downloading the files
from time import sleep # For pause before download
import re # For text cleaning
from pypdf import PdfReader # For reading the PDF files
from sklearn.feature_extraction.text import TfidfVectorizer # For a simple retriever
from sklearn.metrics.pairwise import cosine_similarity # For similarity search

🦥 Unsloth: Will patch your computer to enable 2x faster free finetuning.
Unsloth: Your Flash Attention 2 installation seems to be broken. Using Xformers instead. No performance changes will be seen.
🦥 Unsloth Zoo will now patch everything to make training faster!


The cell below handles the filepath of the input and output files.

In [6]:
PROJECT_CWD = Path.cwd() # Reading the current working directory
DATASET_DIR = PROJECT_CWD / input_dataset # Setting the path for the dataset
OUTPUT_DIR = PROJECT_CWD / output_dataset # Setting path for the output dataset
CONTEXT_DIR = PROJECT_CWD / "context" # Folder containing the PDF files
ADAPTER_DIR = PROJECT_CWD / "model2_lora_adapter" # Path for the LoRA adapter from model2

Now, the dataset will be read into a Pandas dataframe for easier handling.

In [7]:
# Importing the dataset as a Pandas-dataframe
df = pd.read_csv(filepath_or_buffer = DATASET_DIR) # Reading the dataset file

### Task 3.3: RAG - Augmeting answer generation with Austrian Legal Text
In Task 1, each group had to formulate questions and answers (with sources) regarding the Austrian Tax Law, with each group having a different focus. Our grouped focused on international taxation.
In this task, the Large Language model should ve given additional information from Austrian Legal Text.

To configure the basic functionality, we will first configure some parameters.

In [8]:
# RAG settings
CHUNK_SIZE_WORDS = 180 # Length of one retrieved chunk
CHUNK_OVERLAP_WORDS = 40 # Overlap between neighbouring chunks
TOP_K_CHUNKS = 3 # Number of retrieved chunks per question

# Inference settings
MODEL_NAME = "unsloth/gemma-3-4b-it-unsloth-bnb-4bit" # Define model name
MAX_SEQ_LENGTH = 2048 # Setting the maximal context length
MAX_NEW_TOKENS = 512 # Setting the amount of new tokens generated
BATCH_SIZE = 8 # Setting batch size
DTYPE = None # Let computer choose FP accuracy
LOAD_IN_4_BIT = True # Loading in 4 bit to reduce memory usage
SEED = 30112003 # Seed for reproduceability

# Setting seed (depends on CUDA availability, which is available in Google Colab)
torch.manual_seed(SEED)
if torch.cuda.is_available():
    torch.cuda.manual_seed_all(SEED)

Now, we will configure the RAG pipeline. For this, we will first get the names of all available PDF files.

In [9]:
PDF_FILES = sorted(CONTEXT_DIR.glob("*.pdf")) # Read all PDF names from the context folder

if not PDF_FILES:
  raise FileNotFoundError(
      f"""No PDF files were found in {CONTEXT_DIR.resolve()}.
      Create a subfolder named "context" and place the PDFs with Austrian Legal Text in it"""
  )

Now we will define a function that will clean the text from the PDFs using regular expressions.

In [10]:
def clean_text(text: str) -> str:
  return re.sub(r"\s+", " ", str(text)).strip() # Remove whitespace from extracted PDF text

Now we will define a function that splits the text into chunks.

In [11]:
def split_into_chunks(text: str, chunk_size: int = CHUNK_SIZE_WORDS, overlap: int = CHUNK_OVERLAP_WORDS):
  words = clean_text(text).split() # Split text into words

  # If no words are present
  if not words:
    return [] # ...return a empty list

  step = max(chunk_size - overlap, 1) # Ensure the step is atleast 1
  chunks = [] # Empty list for text chunks

  for start in range(0, len(words), step): # Iterate over word list, moving by step each time
    chunk_words = words[start : start + chunk_size] # Slice relevant words
    if not chunk_words: # If no words are there
      continue
    chunks.append(" ".join(chunk_words)) # Join words together into one chunk
    if start + chunk_size >= len(words): # If chunk is end of text
      break

  return chunks # Return chunks

In [12]:
def build_rag_index(pdf_files):
  context_chunks = [] # Empty list for context chunks

  for pdf_file in pdf_files: # Open each PDF file
    reader = PdfReader(str(pdf_file)) # Create reader object

    for page_number, page in enumerate(reader.pages, start = 1): # Iterate over each page
      page_text = clean_text(page.extract_text() or "") # Clean page text, if no text return ""

      if not page_text: # If page is empty
        continue

      for chunk in split_into_chunks(page_text): # Split page text into chunks and iterate over them
        # Append chunk with file name, page number, and text
        context_chunks.append({
            "source": pdf_file.name,
            "page": page_number,
            "text": chunk
        })

  # Check if there are usable chunks
  if not context_chunks:
    raise ValueError(f"No readable text in context files.")

  # Vectorize chunks; use unigrams and bigrams for better context (e. g. phrases)
  vectorizer = TfidfVectorizer(ngram_range = (1, 2))

  # Scan all chunks, learn vocabulary and convert to TF-IDF vector
  context_matrix = vectorizer.fit_transform([chunk["text"] for chunk in context_chunks])

  # Return metadata + text, vectorizer, and context matrix
  return context_chunks, vectorizer, context_matrix


Now we will define a function that will find the most relevant chunks for a given question.

In [13]:
def retrieve_context(question: str, top_k: int = TOP_K_CHUNKS) -> str:
  query = question.strip() # Clean question text

  if not query: # If question/query is empty
    return "Kein zusätzlicher Kontext gefunden."

  query_vector = CONTEXT_VECTORIZER.transform([query]) # Convert query into a vector
  similarities = cosine_similarity(query_vector, CONTEXT_MATRIX).ravel() # Compute cosine similarity between question vector and chunk vectors; .ravel flattens

  ranked_indices = similarities.argsort()[::-1] # Sort chunk indices by similarity score in descending order

  selected_contexts = [] # Empty list for selected contexts

  for idx in ranked_indices: # Loop through chunk indices from best to worst match
    if similarities[idx] <= 0: # If score is negative or zero
      continue # Skip chunk

    chunk = CONTEXT_CHUNKS[idx] # Get chunk text from current index
    # Format chunk
    selected_contexts.append(
            f"[Quelle: {chunk['source']}, Seite {chunk['page']}]\n{chunk['text']}"
        )

    # Break if k is reached
    if len(selected_contexts) >= top_k:
      break

    # If selected chunk list is empty, return "no additional context found "
    if not selected_contexts:
      return "Kein zusätzlicher Kontext gefunden."

  # Return joined contexts
  return "\n\n".join(selected_contexts)

Now we run the index-building function and store the outputs in global variables used later by retrieval.

In [14]:
CONTEXT_CHUNKS, CONTEXT_VECTORIZER, CONTEXT_MATRIX = build_rag_index(PDF_FILES) # Build objects

Now we will load and configure the model and tokenizer for further use. We start by getting the `model` and `tokenizer` objects. For this step, we will use the fine-tuned `model2`.

In [15]:
model, tokenizer = FastLanguageModel.from_pretrained( # Load model and tokenizer
    model_name = str(ADAPTER_DIR), # Model which should be used; here the fine-tuned model2
    max_seq_length = MAX_SEQ_LENGTH, # Maximal context length
    dtype = DTYPE, # Dtype preference; none in this case; determines precision
    load_in_4bit = LOAD_IN_4_BIT # Load model in 4 bit for better hardware performance
)

==((====))==  Unsloth 2026.4.4: Fast Gemma3 patching. Transformers: 5.5.0.
   \\   /|    NVIDIA L4. Num GPUs = 1. Max memory: 22.034 GB. Platform: Linux.
O^O/ \_/ \    Torch: 2.10.0+cu128. CUDA: 8.9. CUDA Toolkit: 12.8. Triton: 3.6.0
\        /    Bfloat16 = TRUE. FA [Xformers = 0.0.35. FA2 = False]
 "-____-"     Free license: http://github.com/unslothai/unsloth
Unsloth: Fast downloading is enabled - ignore downloading bars which are red colored!


model.safetensors:   0%|          | 0.00/4.56G [00:00<?, ?B/s]

Loading weights:   0%|          | 0/883 [00:00<?, ?it/s]

generation_config.json:   0%|          | 0.00/210 [00:00<?, ?B/s]

processor_config.json:   0%|          | 0.00/70.0 [00:00<?, ?B/s]

...

preprocessor_config.json:   0%|          | 0.00/570 [00:00<?, ?B/s]

tokenizer_config.json: 0.00B [00:00, ?B/s]

tokenizer.json:   0%|          | 0.00/33.4M [00:00<?, ?B/s]

added_tokens.json:   0%|          | 0.00/35.0 [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/670 [00:00<?, ?B/s]

Now we will prepare the model for inference.

In [16]:
FastLanguageModel.for_inference(model) # Switches into inference mode

PeftModelForCausalLM(
  (base_model): LoraModel(
    (model): Gemma3ForConditionalGeneration(
      (model): Gemma3Model(
        (vision_tower): SiglipVisionModel(
          (vision_model): SiglipVisionTransformer(
            (embeddings): SiglipVisionEmbeddings(
              (patch_embedding): Conv2d(3, 1152, kernel_size=(14, 14), stride=(14, 14), padding=valid)
              (position_embedding): Embedding(4096, 1152)
            )
            (encoder): SiglipEncoder(
              (layers): ModuleList(
                (0-26): 27 x SiglipEncoderLayer(
                  (layer_norm1): LayerNorm((1152,), eps=1e-06, elementwise_affine=True)
                  (self_attn): SiglipAttention(
                    (k_proj): lora.Linear(
                      (base_layer): Linear(in_features=1152, out_features=1152, bias=True)
                      (lora_dropout): ModuleDict(
                        (default): Identity()
                      )
                      (lora_A): ModuleDict(
  

In [17]:
model.eval() # Switches into evaluation mode

PeftModelForCausalLM(
  (base_model): LoraModel(
    (model): Gemma3ForConditionalGeneration(
      (model): Gemma3Model(
        (vision_tower): SiglipVisionModel(
          (vision_model): SiglipVisionTransformer(
            (embeddings): SiglipVisionEmbeddings(
              (patch_embedding): Conv2d(3, 1152, kernel_size=(14, 14), stride=(14, 14), padding=valid)
              (position_embedding): Embedding(4096, 1152)
            )
            (encoder): SiglipEncoder(
              (layers): ModuleList(
                (0-26): 27 x SiglipEncoderLayer(
                  (layer_norm1): LayerNorm((1152,), eps=1e-06, elementwise_affine=True)
                  (self_attn): SiglipAttention(
                    (k_proj): lora.Linear(
                      (base_layer): Linear(in_features=1152, out_features=1152, bias=True)
                      (lora_dropout): ModuleDict(
                        (default): Identity()
                      )
                      (lora_A): ModuleDict(
  

Now we want to make sure that the model uses the correct architecture.

In [18]:
DEVICE = next(model.parameters()).device # Checking which device torch runs on
DEVICE

device(type='cuda', index=0)

Now we will configure a system prompt. With this, the LLM knows how to "behave". This system prompt is adapted to prefer provided context.

In [19]:
SYSTEM_PROMPT = """Du bist ein Assistent für österreichisches Steuerrecht.
Beantworte Fragen präzise, knapp und fachlich, wenn möglich in maximal 4-5 Sätzen, korrekt auf Deutsch.
Nutze vorrangig den bereitgestellten Rechtskontext. Wenn der Kontext nicht ausreicht, sage das offen. Wenn die Rechtslage unklar ist oder dir Informationen fehlen, sage das offen.
Erfinde keine Gesetzesstellen oder Fakten. Zitiere verwendetete Paragraphen und Richtlinien des österreichen Rechts nach gültigen Zitierregeln.
"""

Now, we will define a `build_prompt_question`-function that will create the prompt for a specific question.

In [20]:
def build_prompt_question(question: str, context: str = ""): # Formats the prompts for use
    question_text = str(question).strip() # Get cleaned question text
    # Get cleaned context text
    context_text = str(context).strip() if context else "Kein zusätzlicher Kontext gefunden."

    user_prompt = (
        f"Frage: \n{question_text}\n\n"
        f"Zusätzlicher Rechtskontext:\n{context_text}"
    )

    # As the model is multimodal, the type of the input has to be specified
    return [
        {"role": "system",
         "content": [{"type": "text", "text": SYSTEM_PROMPT}]},
        {"role": "user",
         "content": [{"type": "text", "text": user_prompt}]}
    ]

Now we will define a function that will answer the questions in a batched manner.

In [21]:
def answer_questions_batched(questions, batch_size: int = BATCH_SIZE): # Function to answer the questions
    questions = list(questions) # Make sure "questions" is a list
    retrieved_contexts = [retrieve_context(q) for q in questions] # Retrieve context for all questions
    conversations = [build_prompt_question(q, c) for q, c in zip(questions, retrieved_contexts)] # Build prompt for every question
    answers = [None] * len(conversations) # Create None-list with length of no. of questions

    # Apply padding
    ## Some tokenizers have the tokenizer in a additional attribute
    if hasattr(tokenizer, "tokenizer"): # If in additional attribute
        tokenizer.tokenizer.padding_side = "left" # Apply left padding
        if tokenizer.tokenizer.pad_token is None: # If Padding Token is None
            tokenizer.tokenizer.pad_token = tokenizer.tokenizer.eos_token # Set Padding Token to EOS token
        eos_id = tokenizer.tokenizer.eos_token_id # Set EOS ID
    else:
        tokenizer.padding_side = "left" # Apply left padding
        if tokenizer.pad_token is None: # If no padding token is set
            tokenizer.pad_token = tokenizer.eos_token # Set Padding token to EOS token
        eos_id = tokenizer.eos_token_id # Set EOS ID

    # This batches the question answering
    for batch_start in range(0, len(conversations), batch_size): # Range from 0 to n with batch_size spacing
        batch_convs = conversations[batch_start: batch_start + batch_size] # Get conversations for batch

        # Apply chat template to batch
        inputs = tokenizer.apply_chat_template(
            batch_convs, # The conversations
            tokenize = True, # Tokenizes the conversations
            add_generation_prompt = True, # Adds marker for completion
            return_dict = True, # Request dictionairy output; for accessing input_ids and attention_mask
            return_tensors = "pt", # Return PyTorch Tensors
            padding = True, # Apply padding (left as set before)
            truncation = True, # Cut off when too long
        ).to(DEVICE) # Send to device

        input_len = inputs["input_ids"].shape[1] # Get size of a answer (all same length due to padding)

        # Actually answer the questions
        ## Inference mode
        with torch.inference_mode():
            outputs = model.generate(
                **inputs, # Apply inputs from tokenizer
                max_new_tokens = MAX_NEW_TOKENS, # Maximum no. of new tokens
                do_sample = False, # Greedy decoing, better for reproduceability
                use_cache = True, # Use caching for better performance
                pad_token_id = eos_id, # Set Padding Token
                eos_token_id = eos_id, # Set EOS token
            )

        generated_only = outputs[:, input_len:] # Only keep generated answers
        decoded = tokenizer.batch_decode( # Decode answers
            generated_only,
            skip_special_tokens = True,
        )

        for i, answer in enumerate(decoded):# Go through decoded answers
            answers[batch_start + i] = answer.strip() # Clean answer and add to answers list with right index

    return answers # Return all answers

Now that we have defined to logic, we can combine the functions to start the generation of the answers.

In [22]:
df_prompts = df["prompt"].tolist() # Convert prompts-column to list

In [23]:
prompt_answers = answer_questions_batched(df_prompts, batch_size = BATCH_SIZE) # Generate answers in batches

Kwargs passed to `processor.__call__` have to be in `processor_kwargs` dict, not in `**kwargs`
Kwargs passed to `processor.__call__` have to be in `processor_kwargs` dict, not in `**kwargs`
Kwargs passed to `processor.__call__` have to be in `processor_kwargs` dict, not in `**kwargs`
Kwargs passed to `processor.__call__` have to be in `processor_kwargs` dict, not in `**kwargs`
Kwargs passed to `processor.__call__` have to be in `processor_kwargs` dict, not in `**kwargs`
Kwargs passed to `processor.__call__` have to be in `processor_kwargs` dict, not in `**kwargs`
Kwargs passed to `processor.__call__` have to be in `processor_kwargs` dict, not in `**kwargs`
Kwargs passed to `processor.__call__` have to be in `processor_kwargs` dict, not in `**kwargs`
Kwargs passed to `processor.__call__` have to be in `processor_kwargs` dict, not in `**kwargs`
Kwargs passed to `processor.__call__` have to be in `processor_kwargs` dict, not in `**kwargs`
Kwargs passed to `processor.__call__` have to be i

Now we can add the generated answers back to the dataframe.

In [24]:
df["answer"] = prompt_answers # Add answers back to dataframe

The ID and answer columns can now be exported.

In [25]:
# Output only the desired columns ID and answer
df[["id", "answer"]].to_csv(OUTPUT_DIR, # to set output directory
                            index = False, # No index wanted
                            quotechar = '"') # Wrap cell text in quotes

Now we wait for a few seconds until Colab has processed the file.

In [28]:
sleep(10) # Wait for 10 seconds until Colab has processed the file

Now we can download the results.

In [27]:
files.download(OUTPUT_DIR.resolve()) # Automatically download file

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>